# Dependence measures

The dependence screen is plumbing with a disclosure policy: it measures
three coefficients on every strong numeric pair, keeps the top 20, and
labels what it found as an *association* — `associations/statistical`, a
category deliberately weaker than a dependency. Association says two
columns move together; it does not say one determines the other.

## Three coefficients

- **Pearson** — linear association, the cheapest signal.
- **Spearman** — monotone association, computed on midranks (ties
  averaged), so a monotone-but-bending relationship scores 1 while its
  Pearson coefficient does not.
- **Chatterjee's ξ** (JASA 2021) — an asymmetric, rank-based
  measure of "Y is a function of X": one sort plus a rank walk, O(n log
  n), robust to arbitrary (not just monotone) functional shapes. The
  screen computes it in both directions, because `ξ(x → y)` and
  `ξ(y → x)` answer different questions.

ξ needs care with ties in x: Chatterjee's estimator breaks x-ties
uniformly at random, and the randomization is seeded — a fixed ordering
would correlate with row position and bias the statistic badly (a block
id against a row counter can otherwise report ξ ≈ 0.99 on independent
data). Constants on either side make ξ undefined, as does n < 30; the
screen returns nothing rather than a degenerate zero.

## Screening and caps

All numeric pairs are screened first — a cheap normalized mutual
information over the encoded bins on up to 2,048 rows — and the
strongest 496 pairs (2,016 on `deep`) get the full three-coefficient
statistics over at most 20,000 (50,000) pair-complete sampled rows.
Findings survive when the strongest signal reaches 0.5 (or |Pearson| is
exactly 1), the top 20 are kept, and pairs with signal ≥ 0.98 become
redundancy links that feed the connected-groups view. Every finding
carries all four numbers, the pair-complete n, and the sampling flag;
the basis is `estimated` — a sample statistic, disclosed as one.

In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(31)
n = 4_000
x = rng.uniform(-1, 1, n)
df = pl.DataFrame({
    "x": x,
    "linear": 3 * x + 1,            # Pearson ~ 1
    "monotone_bend": x ** 3,        # Spearman 1, Pearson < 1
    "parabola": x ** 2,             # xi high, Pearson ~ 0
})

report = onceover.profile(df)
for f in report.findings:
    if f["category"] == "associations":
        m = f["evidence"][0]["metrics"]
        a, b = m["variables"]
        print(f"{a:>14s} × {b:<14s} pearson={m['pearson']:+.3f} "
              f"spearman={m['spearman']:+.3f} xi(a→b)={m['xi_ab']:.3f} xi(b→a)={m['xi_ba']:.3f}")

Reading the output: the linear pair scores on everything; the cubic pair
keeps Spearman at 1 while Pearson dips (bending, not noise); the parabola
is invisible to Pearson and Spearman — it is not a function of x in the
monotone sense — but ξ(x → parabola) recovers it, and the two ξ
directions disagree exactly as they should: `x` determines `x²`, but
`x²` does not determine `x`.

## What a coefficient is not

An association is not a mapping — that claim belongs to the
functional-dependency pass, with canonical verification behind it. An
association is not causal — no coefficient here is. And ξ near zero on
a sample is not proof of independence; it is the absence of detected
structure at that sample size, a statement the coverage block bounds
([](../usage/coverage.ipynb)).